# Estimating the number of chat users

The analysis ran locally with private course inputs. This preview shows selected aggregate results from that check. The course analysis cells below have had their original outputs and execution counts cleared.

## Verified results

With the recovered training workbook, the regression reproduced the archived mean squared error (17,175.41) and R² (0.84242). The rerun also matched the ten predictions displayed in the original notebook preview. The separately archived CSV contains different predictions for the same 1,438 input rows.

In [ ]:
from pathlib import Path
import os

# Jupyter may start the kernel in the notebook directory or in the repository root.
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
DATA_DIR = Path(os.getenv("TWITCH_MODERATION_DATA") or repo_root / "data" / "private").expanduser()

import pandas as pd
creator_b = pd.read_excel(DATA_DIR / "support/training_audience_b.xlsx")
creator_b = creator_b[['Data', 'Spettatori medi', 'Follower', 'Minuti di streaming','Spettatori max', 'Spettatori unici', 'Utenti chat']]
creator_b.columns = ['Data', 'AvgViewers', 'followers', 'MinStreaming', 'MaxViewers', 'UniqueViewers', 'ChatUsers']
creator_b['Data'] =  pd.to_datetime(creator_b['Data'])
creator_b.set_index('Data', drop=True, inplace=True)


In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
df = creator_b.copy()
# Define the columns you want to visualize
columns_to_visualize = ['AvgViewers', 'followers', 'MinStreaming', 'MaxViewers', 'UniqueViewers']

# Define a function to remove upper outliers
def remove_upper_outliers(df, column, quantile):
    threshold = df[column].quantile(quantile)
    return df[df[column] <= threshold]

# Loop through each column and create scatter plots removing upper outliers
for column in columns_to_visualize:
    plt.figure(figsize=(8, 6))
    df_no_outliers = remove_upper_outliers(df, column, 0.99)  # Remove top 1% of data
    sns.scatterplot(x=column, y='ChatUsers', data=df_no_outliers)
    plt.title(f'Scatter plot of {column} vs ChatUsers (without upper outliers)')
    plt.xlabel(column)
    plt.ylabel('ChatUsers')
    plt.show()


In [ ]:
df = creator_b.copy()

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error ,r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
# Step 1: Remove top 1 percent outliers from the original data
def remove_upper_outliers(df, column, quantile):
    threshold = df[column].quantile(quantile)
    return df[df[column] <= threshold]

df_no_outliers = df.copy()  # Copy the original DataFrame
# Assuming df is your DataFrame with a datetime index
df_no_outliers['Year'] = df_no_outliers.index.year
df_no_outliers['Month'] = df_no_outliers.index.month
df_no_outliers['Day'] = df_no_outliers.index.day

for column in columns_to_visualize:
    df_no_outliers = remove_upper_outliers(df_no_outliers, column, 0.99)


scaler = StandardScaler()
scaled_features_df1 = pd.DataFrame(scaler.fit_transform(df_no_outliers[['AvgViewers', 'followers', 'MinStreaming', 'MaxViewers']])
                        , columns=['AvgViewers', 'followers', 'MinStreaming', 'MaxViewers'])
# Combine scaled features with unscaled 'Year', 'Month', 'Day' columns
scaled_df1 = pd.concat([scaled_features_df1, df_no_outliers[['Year', 'Month', 'Day']].reset_index(drop = True)], axis=1)



# Step 2: Split the data into features (X) and target variable (y)
X = scaled_df1[['AvgViewers', 'followers', 'MinStreaming', 'MaxViewers', 'Year', 'Month', 'Day']]
y = df_no_outliers['ChatUsers']



# Random Forest Regressor
rf_model = LinearRegression()
rf_model.fit(X, y)
rf_y_pred = rf_model.predict(X)
rf_mse = mean_squared_error(y, rf_y_pred)
rf_r2 = r2_score(y, rf_y_pred)
print("Random Forest Regressor:")
print("Mean Squared Error:", rf_mse)
print("R-squared:", rf_r2)

In [ ]:
new = pd.read_excel(DATA_DIR / "support/stream_metrics_d.xlsx")
new['Avg Viewers'] = new['Avg Viewers'].str.replace(',', '')
new['Avg Viewers'] = new['Avg Viewers'].astype(float)
new['Views'] = new['Views'].str.replace(',', '')
new['Views'] = new['Views'].astype(float)
creator_d = new.copy()
creator_d['MinStreaming'] = creator_d['Stream and Duration'].str.extract(r'(\d+\.\d+) hrs').astype(float) * 60
date_pattern = r'(\d{1,2}/[A-Za-z]{3}/\d{4})'
creator_d['Date'] = creator_d['Stream and Duration'].str.extract(date_pattern)
creator_d.set_index('Date', inplace = True)
creator_d.index = pd.to_datetime(creator_d.index)
creator_d = creator_d[['Avg Viewers', 'Max Viewers', 'Followers', 'MinStreaming']]
creator_d.columns = ['AvgViewers', 'MaxViewers', 'followers', 'MinStreaming']
#creator_d['AvgViewers'] = creator_d['AvgViewers'].astype(float)
creator_d['Year'] = creator_d.index.year
creator_d['Month'] = creator_d.index.month
creator_d['Day'] = creator_d.index.day
creator_d = creator_d.reindex(columns=['AvgViewers', 'followers', 'MinStreaming', 'MaxViewers', 'Year', 'Month', 'Day'])


In [ ]:
scaled_features_df2 = scaler.transform(creator_d[['AvgViewers', 'followers', 'MinStreaming', 'MaxViewers']])

# Combine scaled features with unscaled 'Year', 'Month', 'Day' columns for df2
scaled_df2 = pd.concat([pd.DataFrame(scaled_features_df2, columns=['AvgViewers', 'followers', 'MinStreaming', 'MaxViewers']),
                        creator_d[['Year', 'Month', 'Day']].reset_index(drop=True)], axis=1)

In [ ]:
rf_y_pred = rf_model.predict(scaled_df2)
final = creator_d.copy()
final['ChatUsers'] = rf_y_pred
condition = final['MaxViewers'] < final['ChatUsers'] # 80 rows of those
final.loc[condition, 'ChatUsers'] /= 2


In [ ]:
final.to_csv(DATA_DIR / "support/chat_users_predictions_d.csv")

In [ ]:
final['ChatUsers'].plot()

In [ ]:
final.describe()

In [ ]:
creator_b.describe()

In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# Define the columns you want to visualize
columns_to_visualize = ['AvgViewers', 'followers', 'MinStreaming', 'MaxViewers']


# Loop through each column and create scatter plots removing upper outliers
for column in columns_to_visualize:
    plt.figure(figsize=(8, 6))
    #df_no_outliers = remove_upper_outliers(df, column, 0.99)  # Remove top 1% of data
    sns.scatterplot(x=column, y='ChatUsers', data=final)
    plt.title(f'Scatter plot of {column} vs ChatUsers (without upper outliers)')
    plt.xlabel(column)
    plt.ylabel('ChatUsers')
    plt.show()
